# Load Modules

In [1]:
import math
import folium
import pickle
import requests
import numpy as np
import pandas as pd
import geopy.distance
import rustworkx as rx
import geopandas as gpd

from tqdm import tqdm
from scipy.spatial import KDTree
from pretty_warnings import warn
from shapely.geometry import LineString, mapping

# Setup OSRM walking navigation

In [47]:
# docker run -t -v "${PWD}:/data" ghcr.io/project-osrm/osrm-backend osrm-extract -p /opt/foot.lua /data/england.osm.pbf
# docker run -t -v "${PWD}:/data" ghcr.io/project-osrm/osrm-backend osrm-partition /data/england
# docker run -t -v "${PWD}:/data" ghcr.io/project-osrm/osrm-backend osrm-customize /data/england

# docker run -t -i -p 5000:5000 -v "${PWD}:/data" ghcr.io/project-osrm/osrm-backend osrm-routed --algorithm mld /data/england

def get_walking_durations_batch(origin, targets):

    source_coord = f"{origin[1]},{origin[0]}"
    target_coords = [f"{tar[1]},{tar[0]}" for tar in targets]

    all_coords = ";".join([source_coord] + target_coords)
    url = f"http://127.0.0.1:5000/table/v1/foot/{all_coords}?sources=0"

    response = requests.get(url).json()
    return response['durations'][0][1:]

# Load school and map data

In [13]:
# Load school data
school_data = pd.read_csv("data/ofsted_bradford_data.csv")

In [14]:
# Load geospatial data
oa21lsoa = pd.read_excel("data/OA11_LAD21_LSOA11_MSOA11_LEP21_EN_v3.xlsx")
oa_gdf = gpd.read_file("data/Output_Areas_Dec_2011_Boundaries_EW_BGC_2022_-9162807391755440229.geojson" )

# Select only OAs in and around Bradford
selected_oas = oa21lsoa.loc[oa21lsoa["LAD21NM"].isin(["Bradford", "Leeds", "Calderdale", "Kirklees", "Craven", "Harrogate", "Pendle"]), "OA11CD"].unique()
bradford_gdf = oa_gdf.loc[oa_gdf["OA11CD"].isin(selected_oas)]

# Get the centroid latitude and longitude coords for each OA
oa_unique = bradford_gdf.drop_duplicates(subset=["OA11CD"])
centroids_4326 = oa_unique.to_crs(epsg=27700).geometry.centroid.to_crs(epsg=4326)
oa_centroid_dict = dict(zip(oa_unique["OA11CD"], centroids_4326))

# Load bus data

In [15]:
# Load bus data
stops_df = pd.read_csv("data/stops_combined.txt")
stop_times_df = pd.read_csv("data/stop_times_combined.txt")
trips = pd.read_csv("data/trips_combined.txt")

# Covert times to timedelta data types
stop_times_df["arrival_time"] = pd.to_timedelta(stop_times_df["arrival_time"])
stop_times_df["departure_time"] = pd.to_timedelta(stop_times_df["departure_time"])

# Compute useful data features

In [6]:
# # Drop any bus stops which are outside the considered area
# oa_lons = np.array([(oa_centroid_dict[key].x) for key in oa_centroid_dict])
# oa_lats = np.array([(oa_centroid_dict[key].y) for key in oa_centroid_dict])

# stops_df = stops_df.loc[
#     stops_df["stop_lat"].between(oa_lats.min(), oa_lats.max()) &
#     stops_df["stop_lon"].between(oa_lons.min(), oa_lons.max())
# ]

# stop_times_df = stop_times_df.loc[stop_times_df["stop_id"].isin(stops_df["stop_id"])]

In [16]:
# Get a map of stop_ids to their locations
stops_loc_dict = stops_df.set_index("stop_id")[["stop_lat", "stop_lon"]].apply(tuple, axis=1).to_dict()

In [17]:
# Get a map of all stop_ids on each trip
trip_stops_dict = stop_times_df.groupby('trip_id')['stop_id'].apply(set).to_dict()

In [18]:
# Select only morning buses
stop_times_df = stop_times_df[
    stop_times_df["arrival_time"].between(
        pd.Timedelta("07:00:00"),
        pd.Timedelta("10:00:00")
    )
]

# Drop duplicates 8,792,451 -> 2,945,007
stop_times_df = stop_times_df.drop_duplicates(subset=["trip_id", "arrival_time", "departure_time", "stop_sequence"])

In [19]:
# Map the trip id to the route direction
trip_direction_map = trips.set_index("trip_id")["direction_id"].to_dict()
stop_times_df["direction_id"] = stop_times_df["trip_id"].map(trip_direction_map)

# Map each trip to a given route_id
trip_route_id_map = trips.set_index("trip_id")["route_id"].to_dict()
stop_times_df["route_id"] = stop_times_df["trip_id"].map(trip_route_id_map)

In [20]:
# Find the length of each trip based on the number of stops passes
grouped = stop_times_df.groupby("trip_id")["stop_sequence"]
stop_times_df["trip_len"] = (grouped.transform("max") - grouped.transform("min"))

# Isolate longest routes

In [21]:
include_ids = []

# Go through each unique route and find the and the max length trips in each direction
for route_id in tqdm(stop_times_df["route_id"].unique()):
    # Select the routes with this id
    route_trips = stop_times_df.loc[stop_times_df["route_id"] == route_id]

    # Isolate the trips in each direction
    forward_route_trips = route_trips.loc[route_trips["direction_id"] == 0]
    backward_route_trips = route_trips.loc[route_trips["direction_id"] == 1]

    # Find the max length of trips in each direction
    max_trip_len_fore = forward_route_trips["trip_len"].max()
    max_trip_len_back = backward_route_trips["trip_len"].max()

    # Add indexes to list of longest to select later
    include_ids += list(forward_route_trips.loc[forward_route_trips["trip_len"] == max_trip_len_fore, "trip_id"].unique())
    include_ids += list(backward_route_trips.loc[backward_route_trips["trip_len"] == max_trip_len_back, "trip_id"].unique())

longest_trips_df = stop_times_df.loc[stop_times_df["trip_id"].isin(include_ids)].reset_index(drop=True)

100%|██████████| 1182/1182 [00:09<00:00, 122.96it/s]


In [22]:
# Find the mean travel time between each stop
df = longest_trips_df
df = df.sort_values(by=["trip_id", "stop_sequence"])

# Find the time difference between each stop
df["time_from_prev"] = df["arrival_time"] - df.groupby("trip_id")["arrival_time"].shift(1)

In [23]:
# Find the mean travel time between each stop
df = longest_trips_df
df = df.sort_values(by=["trip_id", "stop_sequence"])

# Find the time difference between each stop
df["time_from_prev"] = df["arrival_time"] - df.groupby("trip_id")["arrival_time"].shift(1)

# Set any erroneous negative times to zero
df.loc[df["time_from_prev"] < pd.Timedelta(0), "time_from_prev"] = pd.Timedelta(0)

df["prev_stop_id"] = df.groupby("trip_id")["stop_id"].shift(1)
df["travel_time_seconds"] = df["time_from_prev"].dt.total_seconds()

valid_segments = df.dropna(subset=["prev_stop_id", "travel_time_seconds"])
average_segment_times = valid_segments.groupby(["route_id", "prev_stop_id", "stop_id"])["travel_time_seconds"].mean().reset_index()
mean_times_dict = average_segment_times.set_index(["route_id", "prev_stop_id", "stop_id"])["travel_time_seconds"].to_dict()

# Create Graph

In [48]:
# Assume 8 minute wait to change bus
TRANSFER_PENALTY_SEC = 7 * 60

## Add bus stops, edge weights and route transfer costs

In [ ]:
G = rx.PyDiGraph()

node_to_idx = {}
idx_to_node = {}

def get_or_create_node(node_key):
    if node_key not in node_to_idx:
        stop_id_key = node_key[0] if isinstance(node_key, tuple) else node_key

        lat = stops_loc_dict[stop_id_key][0]
        lon = stops_loc_dict[stop_id_key][1]
            
        idx = G.add_node({"name": node_key, "lat": lat, "lon": lon})
        node_to_idx[node_key] = idx
        idx_to_node[idx] = node_key

    return node_to_idx[node_key]

longest_trips_df = longest_trips_df.sort_values(["trip_id", "stop_sequence"])

# Go through each trip
for trip_id, trip_to_plot in tqdm(longest_trips_df.groupby("trip_id"), desc="Adding bus stop nodes and bus route edges."):
    trip_to_plot = trip_to_plot.reset_index(drop=True)
    route_id = trip_to_plot["route_id"].iloc[0]

    # Go through each stop on the trip
    for i in range(len(trip_to_plot) - 1):
        cur_stop = trip_to_plot.loc[i, "stop_id"]
        next_stop = trip_to_plot.loc[i+1, "stop_id"]
        
        cur_platform = (cur_stop, route_id)
        next_platform = (next_stop, route_id)

        duration_sec = mean_times_dict[(route_id, cur_stop, next_stop)]

        u = get_or_create_node(cur_platform)
        v = get_or_create_node(next_platform)
        
        G.add_edge(u, v, {"weight": float(duration_sec), "type": "transit"})
        
        # Transfer handling
        c_stop_idx = get_or_create_node(cur_stop)
        n_stop_idx = get_or_create_node(next_stop)
        
        if not G.has_edge(c_stop_idx, u):
            G.add_edge(c_stop_idx, u, {"weight": float(TRANSFER_PENALTY_SEC), "type": "transfer"})
        if not G.has_edge(n_stop_idx, v):
            G.add_edge(n_stop_idx, v, {"weight": float(TRANSFER_PENALTY_SEC), "type": "transfer"})
            
        if not G.has_edge(u, c_stop_idx):
            G.add_edge(u, c_stop_idx, {"weight": 0.0, "type": "exit"})
        if not G.has_edge(v, n_stop_idx):
            G.add_edge(v, n_stop_idx, {"weight": 0.0, "type": "exit"})

## Define useful data

In [50]:
# Isolate the stops that exist as node on the current graph
stop_id_nodes_on_graph = [key for key in node_to_idx.keys() if isinstance(key, str)]
stop_locs_df = pd.DataFrame({"stop_id": stop_id_nodes_on_graph})

stop_coords = stop_locs_df["stop_id"].map(stops_loc_dict).tolist()
stop_locs_df["stop_loc"] = stop_coords

In [51]:
# Define fast approximate heuristic for finding nearest stops
LON_SCALE_FACTOR = math.cos(math.radians(53.8))

In [52]:
# Create stop KDTree

# Apply LON_SCALE_FACTOR directly to the coordinates for the spatial tree
scaled_stop_coords = [(lat, lon * LON_SCALE_FACTOR) for lat, lon in stop_coords]

# Build Tree
stop_tree = KDTree(scaled_stop_coords)

In [53]:
# Define number of stop neighbours to consider
N_NEIGHBOURS = 100

## Add walks between stops

In [54]:
# Query nearest neighbours
distances, indices = stop_tree.query(scaled_stop_coords, k=N_NEIGHBOURS+1)

seen_walk_edges = set()

for i, stop_idx_list in enumerate(tqdm(indices, desc="Adding walks between nodes")):
    origin_stop = stop_id_nodes_on_graph[i]
    origin_node = node_to_idx[origin_stop]
    
    # Skip the first result as it's the origin stop itself (distance 0)
    neighbour_ids = [stop_id_nodes_on_graph[j] for j in stop_idx_list[1:]]
    
    times = get_walking_durations_batch(stop_coords[i], [stop_coords[j] for j in stop_idx_list[1:]])
    
    for neighbor_stop, walk_time in zip(neighbour_ids, times):
        if walk_time is not None:
            neighbor_node = node_to_idx[neighbor_stop]
            
            # Create a unique identifier for this pair to avoid duplicates
            edge_pair = frozenset([origin_node, neighbor_node])
            
            if edge_pair not in seen_walk_edges:
                G.add_edge(origin_node, neighbor_node, {"weight": walk_time, "type": "walk"})
                G.add_edge(neighbor_node, origin_node, {"weight": walk_time, "type": "walk"})
                seen_walk_edges.add(edge_pair)

Adding walks between nodes: 100%|██████████| 30039/30039 [09:13<00:00, 54.28it/s]


## Add schools to graph

In [55]:
school_data = school_data.dropna(subset=["Lat"])

In [60]:
N_NEIGHBOURS_SCHOOLS = 500

for i, row in tqdm(school_data.iterrows(), total=len(school_data), desc="Connecting Schools"):
    # Add school as point on graph
    urn = row["URN"]
    lat, lon = row["Lat"], row["Lon"]
    
    idx = G.add_node({"name": urn, "lat": lat, "lon": lon})
    node_to_idx[urn] = idx
    idx_to_node[idx] = urn

    # Find N nearest bus stops
    scaled_school = (lat, lon * LON_SCALE_FACTOR)
    distances, indices = stop_tree.query(scaled_school, k=N_NEIGHBOURS_SCHOOLS)
    
    nearest_stop_ids = [stop_id_nodes_on_graph[j] for j in indices]
    nearest_stop_coords = [stop_coords[j] for j in indices]

    # Calculate walk times
    walk_times = get_walking_durations_batch((lat, lon), nearest_stop_coords)

    # Add walk edges from each stop to the school (Notice the edge direction is reversed: stop -> school)
    for stop_id, walk_time in zip(nearest_stop_ids, walk_times):
        if walk_time is not None:
            G.add_edge(node_to_idx[stop_id], idx, {"weight": walk_time, "type": "walk"})

Connecting Schools: 100%|██████████| 104/104 [00:09<00:00, 10.66it/s]


## Add OA centroids as graph nodes

In [62]:
N_NEIGHBOURS_HOME_OA = 200

school_locations = school_data.apply(lambda f: (f["Lat"], f["Lon"]), axis=1).to_list()
school_urns = school_data["URN"].to_list()

for oa, point in tqdm(oa_centroid_dict.items(), desc="Connecting OAs"):
    # Add oa centroid as node on graph
    lon, lat = point.x, point.y

    idx = G.add_node({"name": oa, "lat": lat, "lon": lon})
    node_to_idx[oa] = idx
    idx_to_node[idx] = oa
    
    # Find N nearest bus stops
    scaled_oa = (lat, lon * LON_SCALE_FACTOR)
    distances, indices = stop_tree.query(scaled_oa, k=N_NEIGHBOURS_HOME_OA)
    
    nearest_stop_ids = [stop_id_nodes_on_graph[j] for j in indices]
    nearest_stop_coords = [stop_coords[j] for j in indices]

    # Calculate walk times 
    walk_times = get_walking_durations_batch((lat, lon), nearest_stop_coords)

    # Add walk edges from oa centroid to nearest N stops
    for stop_id, walk_time in zip(nearest_stop_ids, walk_times):
        G.add_edge(idx, node_to_idx[stop_id], {"weight": walk_time, "type": "walk"})


    # Find walk times to all schools
    school_walk_times = get_walking_durations_batch((lat, lon), school_locations)

    # Add walks from OA to each school (ensures route never recommends a bus longer than just walking)
    for school_urn, walk_time in zip(school_urns, school_walk_times):
        if walk_time is not None:
            G.add_edge(idx, node_to_idx[school_urn], {"weight": walk_time, "type": "walk"})

Connecting OAs: 100%|██████████| 7070/7070 [12:05<00:00,  9.74it/s]


### Save graph

In [63]:
with open("transit_graph_.pkl", "wb") as f:
    pickle.dump(G, f)

with open("node_to_idx_.pkl", "wb") as f:
    pickle.dump(node_to_idx, f)

## Draw Graph

In [59]:
raise Exception("Stop here when running all.")

Exception: Stop here when running all.

### Useful data

In [ ]:
urn_name_map = school_data.set_index("URN")["School name"].to_dict()

In [ ]:
# Get nodes within limited Bradford area
def is_in_bradford(lat, lon): return (53.71 <= lat <= 53.92 and -1.92 <= lon <= -1.66)

bradford_node_indices = {idx for idx in G.node_indices() if is_in_bradford(G[idx]["lat"], G[idx]["lon"])}

In [ ]:
features = []

for u, v, data in G.weighted_edge_list():
    if u in bradford_node_indices and v in bradford_node_indices:
        if isinstance(data, dict) and data.get("type") == "walk":
            u_node, v_node = G[u], G[v]

            # GeoJSON specification requires [longitude, latitude] (x, y)
            line = LineString([
                (u_node["lon"], u_node["lat"]),
                (v_node["lon"], v_node["lat"]),
            ])

            # Extract route_id if using platform tuple format: (stop_id, route_id)
            route_id = u_node["name"][1] if isinstance(u_node["name"], tuple) else "Transit"

            features.append({
                "type": "Feature",
                "geometry": mapping(line)
            })

geojson_data = {
    "type": "FeatureCollection",
    "features": features,
}

### Walk map

In [ ]:
m = folium.Map(location=[53.7959, -1.7594], zoom_start=12, tiles="Cartodb Positron")

folium.GeoJson(
    geojson_data,
    name="Bus Transit Network",
    style_function=lambda feature: {
        "color": "#163454",
        "weight": 1,
        "opacity": 0.6,
    }
).add_to(m)

m.save("output/all_edges_geojson.html")

In [ ]:
m = folium.Map(location=[53.7959, -1.7594], zoom_start=12, tiles="Cartodb Positron")

for idx in tqdm(G.node_indices(), desc="Drawing nodes"):
    node = G[idx]
    if is_in_bradford(node["lat"], node["lon"]):
        folium.CircleMarker(
            location=(node["lat"], node["lon"]),
            radius=1,
            color="#202F8E",
            weight=1,
            fill=True,
            fill_opacity=0.9
        ).add_to(m)

m.save("output/all_graph_nodes.html")

Drawing nodes: 100%|██████████| 109913/109913 [00:00<00:00, 213899.27it/s]


In [ ]:
m = folium.Map(location=[53.7959, -1.7594], zoom_start=12, tiles="Cartodb Positron")

for urn in school_data["URN"].unique():
    idx = node_to_idx[urn]
    node = G[idx]
    folium.CircleMarker(
        location=(node["lat"], node["lon"]),
        radius=4,
        color="#C20000",
        fill=True,
        fill_colour="#CB4A4A",
        weight=1,
        fill_opacity=0.9,
        popup=f"{urn_name_map[urn]}",
    ).add_to(m)

for oa in oa_centroid_dict:
    idx = node_to_idx[oa]
    node = G[idx]
    folium.CircleMarker(
        location=(node["lat"], node["lon"]),
        radius=2,
        color="#E4E246",  # Dark blue
        weight=1,
        fill=True,
        fill_opacity=0.9,
        popup=f"OA Centroid: {oa}",
    ).add_to(m)

for idx in G.node_indices():
    if is_in_bradford(G[idx]["lat"], G[idx]["lon"]):
        if idx_to_node[idx] in stop_id_nodes_on_graph:
                node = G[idx]
                folium.CircleMarker(
                    location=(node["lat"], node["lon"]),
                    radius=2,
                    color="#465BE4",  # Dark blue
                    weight=1,
                    fill=True,
                    fill_opacity=0.9,
                    popup=f"Bus stop: {idx_to_node[idx]}",
                ).add_to(m)
    
m.save("output/school_graph.html")